# Example planners
A motion planner decides **where the ego vehicle should drive and how it should move**
among other vehicles. We use one benchmark, the Erlangen junction turn
`DEU_ErlangenCRC26P-12_17300_I-1`, to introduce traffic prediction, curvilinear
coordinates and routing. We then use these ideas to understand a constant-speed
dummy planner and the reactive planner.

Use the competition environment and select **Restart Kernel and Run All**.
The first four sections introduce the planning components and construct a simple
trajectory. The final section runs a SUMO simulation and reuses its result for
animation and export.


In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
from commonroad.common.file_reader import CommonRoadFileReader
from commonroad.visualization.mp_renderer import MPRenderer
from IPython.display import Image, display

from planner.common.scenarios import ASSETS

starter = ASSETS / "scenarios/public/DEU_ErlangenCRC26P-12_17300_I-1.xml"
scenario, problems = CommonRoadFileReader(str(starter)).open()
problem = next(iter(problems.planning_problem_dict.values()))
current = problem.initial_state.time_step
print(f"Time step: {scenario.dt} s; traffic vehicles: {len(scenario.dynamic_obstacles)}")


The following helpers handle plotting. `draw_scene()` shows the map, current traffic,
ego start and goal, while hiding recorded future trajectories. `show_plot()` displays
the figures in both native and Docker notebooks.


In [ ]:
def draw_scene(title):
    fig, ax = plt.subplots(figsize=(9, 5))
    renderer = MPRenderer(ax=ax)
    renderer.draw_params.time_begin = current
    renderer.draw_params.time_end = current
    renderer.draw_params.dynamic_obstacle.trajectory.draw_trajectory = False
    renderer.draw_params.planning_problem.initial_state.state.draw_arrow = False
    renderer.draw_params.planning_problem.initial_state.state.radius = 0.4
    scenario.draw(renderer)
    problem.draw(renderer)
    renderer.render()
    ax.set(title=title, xlabel="x [m]", ylabel="y [m]")
    return fig, ax


def show_plot(fig):
    image = io.BytesIO()
    fig.savefig(image, format="png", bbox_inches="tight", dpi=120)
    display(Image(data=image.getvalue()))
    plt.close(fig)


## 1. Predict nearby traffic
Knowing another vehicle's current position does not tell us whether it will cross
the ego vehicle's planned path. We also need to estimate how that vehicle will move.
Here we predict the motion of nearby vehicle `30581` over the next two seconds.

[`predict_observations()`](../planner/common/prediction.py) predicts each vehicle's
positions along its lane at constant longitudinal speed, preserving its lateral
offset. Both the search and reactive examples use this helper. It relies on current
observations and map geometry, without reading recorded future motion or SUMO's
intended routes.


In [ ]:
from planner.common.prediction import predict_observations

horizon = 20  # 20 steps * 0.1 s = 2 seconds
prediction = predict_observations(scenario, current, horizon)
actor = prediction.obstacle_by_id(30581)
forecast_states = [actor.initial_state, *actor.prediction.trajectory.state_list]
forecast_xy = np.array([state.position for state in forecast_states])
forecast_time = np.array([state.time_step * scenario.dt for state in forecast_states])

fig, ax = draw_scene("Two-second forecast for vehicle 30581")
ax.plot(forecast_xy[:, 0], forecast_xy[:, 1], color="tab:orange", linewidth=2)
points = ax.scatter(
    forecast_xy[:, 0], forecast_xy[:, 1], c=forecast_time, cmap="plasma", zorder=30
)
ax.scatter(*forecast_xy[0], marker="x", color="black", s=80, zorder=31, label="Now")
ax.set_xlim(forecast_xy[:, 0].min() - 5, forecast_xy[:, 0].max() + 5)
ax.set_ylim(forecast_xy[:, 1].min() - 6, forecast_xy[:, 1].max() + 6)
fig.colorbar(points, ax=ax, label="Time ahead [s]")
ax.legend().set_zorder(100)
show_plot(fig)


The colored points show the predicted positions at successive times. This predictor
cannot anticipate braking or lane changes. It follows the next lane when there is
only one successor; at forks or lane ends, it continues in the direction of the
last lane segment. If lane matching or coordinate conversion fails, it predicts
straight motion at the observed speed and heading. Stationary vehicles remain still.

Prediction timestamps count steps from the current observation: `0` means now.
Ego trajectories use absolute simulation steps. For example, at simulation step 30,
prediction step 1 corresponds to ego step 31. Align these times when checking collisions.

To predict movement along a curved lane, we first describe positions relative to
the lane's path. The next section introduces this coordinate system.


## 2. Build a curvilinear coordinate system
Cartesian coordinates `(x, y)` locate a point on the map. Curvilinear, or Frenet,
coordinates locate the same point relative to a reference path:

- **`s`** is the distance along the path from its beginning, in meters.
- **`d`** is the signed lateral distance from the path, in meters. Positive values
  lie to the left when looking along the path's direction.

Points on the reference path have `d = 0`. Increasing `s` while keeping `d` fixed
moves a point along the curve at a constant lateral offset.

We use lanelet `938`, whose direction matches vehicle `30581`'s heading. Because
lanelets overlap at this junction, the predictor uses both heading and position
to choose a lane. The code below resamples its centerline, builds the coordinate
frame, and converts the vehicle's position to `(s, d)` and back to `(x, y)`.


In [ ]:
from commonroad_clcs.pycrccosy import CurvilinearCoordinateSystem
from commonroad_clcs.util import resample_polyline

lane = scenario.lanelet_network.find_lanelet_by_id(938)
lane_reference = resample_polyline(lane.center_vertices, step=0.5)
lane_frame = CurvilinearCoordinateSystem(lane_reference)

observed_xy = actor.initial_state.position
s_actor, d_actor = lane_frame.convert_to_curvilinear_coords(*observed_xy)
recovered_xy = lane_frame.convert_to_cartesian_coords(s_actor, d_actor)
print(f"Vehicle 30581: s = {s_actor:.2f} m, d = {d_actor:.2f} m")
print(f"Cartesian round-trip error: {np.linalg.norm(recovered_xy - observed_xy):.6f} m")


The two plots show the same three points in different coordinates. A and B lie on
the reference path, while C is one meter to its left. In the right plot, their
longitudinal distances and lateral offsets can be read directly from the axes.


In [ ]:
points_sd = np.array([[s_actor, 0.0], [s_actor + 4.0, 0.0], [s_actor + 8.0, 1.0]])
points_xy = np.array([lane_frame.convert_to_cartesian_coords(s, d) for s, d in points_sd])
s_line = np.linspace(s_actor, s_actor + 8.0, 50)
center_xy = np.array([lane_frame.convert_to_cartesian_coords(s, 0.0) for s in s_line])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(center_xy[:, 0], center_xy[:, 1], color="gray", label="Reference path")
axes[1].plot(s_line, np.zeros_like(s_line), color="gray", label="Reference path: d = 0")
colors = ["tab:blue", "tab:orange", "tab:green"]
for label, color, xy, sd in zip("ABC", colors, points_xy, points_sd):
    for ax, point in zip(axes, [xy, sd]):
        ax.scatter(*point, color=color, zorder=3)
        ax.annotate(label, point, xytext=(6, 6), textcoords="offset points")
axes[0].set(title="Cartesian map coordinates", xlabel="x [m]", ylabel="y [m]")
axes[0].set_aspect("equal")
axes[1].set(title="Curvilinear coordinates", xlabel="s [m]", ylabel="d [m]", ylim=(-1, 2))
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.25)
    ax.margins(x=0.12, y=0.15)
fig.tight_layout()
show_plot(fig)


Coordinate conversion works only within the frame's **projection domain**, where
map positions have an unambiguous location relative to the reference path. A planner
must choose a suitable, smooth reference and handle positions outside this domain.

The reactive planner generates ego motion in curvilinear coordinates. SMP instead
connects Cartesian motion primitives, while using the same curvilinear traffic
predictor. Both planners return Cartesian vehicle-center states to the runner.

This local frame describes positions along one lane. To choose which junction exit
leads toward the ego vehicle's goal, we also need a route across lanelets.


## 3. Plan a high-level route
Route planning chooses **which connected lanelets the ego vehicle should follow**
to reach its goal. CommonRoad's route planner finds lanelet sequences, and its
reference-path planner converts a selected route into an `(N, 2)` array of map
coordinates. Choosing speeds and responding to traffic are subsequent planning tasks.

[`StudentPlanner.plan_route()`](../my_submission/custom_planner.py) uses these same two steps.


In [ ]:
from commonroad_route_planner.reference_path_planner import ReferencePathPlanner
from commonroad_route_planner.route_planner import RoutePlanner

routes = RoutePlanner(scenario.lanelet_network, problem).plan_routes()
route = ReferencePathPlanner(
    scenario.lanelet_network, problem, routes
).plan_shortest_reference_path()
reference_path = route.reference_path
print("Selected lanelet route:", route.lanelet_ids)

fig, ax = draw_scene("A route from the ego start toward the goal")
ax.plot(reference_path[:, 0], reference_path[:, 1], color="tab:blue", linewidth=2,
        label="Ego reference path", zorder=25)
ax.scatter(*problem.initial_state.position, color="green", label="Ego now", zorder=26)
ax.legend().set_zorder(100)
show_plot(fig)


The route `[152, 939, 155]` leads into the junction, through the right turn, and
along the outgoing lane. Vehicle `30581` is aligned with lanelet `938`, so its
predicted motion takes a different direction through the junction.

The blue path specifies where to drive. A trajectory must also specify when the
vehicle reaches each position, together with its heading, speed and steering angle.
We start by following the path at the vehicle's current speed.


## 4. Follow the route with a constant-speed dummy
[`StudentPlanner.plan()`](../my_submission/custom_planner.py) calls `plan_route()` and
advances along the returned path by `velocity * scenario.dt` at each step. It preserves
the vehicle's initial displacement from the path to avoid snapping onto it, and stops
at the path's end. The returned `KSState` objects use Cartesian positions and
consecutive absolute simulation steps.

The planner returns five future states by default. Here we request thirty to show
three seconds of motion through the turn. This **geometric plan** illustrates route
following; we have not yet checked whether the vehicle can execute it safely.


In [ ]:
from my_submission.custom_planner import StudentPlanner

dummy = StudentPlanner(horizon_steps=30)
dummy_trajectory = dummy.plan(scenario, problem)
dummy_states = [problem.initial_state, *dummy_trajectory.state_list]
dummy_xy = np.array([state.position for state in dummy_states])
dummy_time = np.array([(state.time_step - current) * scenario.dt for state in dummy_states])

fig, ax = draw_scene("Dummy planner: follow the route at constant speed")
ax.plot(reference_path[:, 0], reference_path[:, 1], "--", color="gray",
        label="Reference path", zorder=25)
ax.plot(dummy_xy[:, 0], dummy_xy[:, 1], "o-", color="tab:blue", markersize=3,
        label=f"Ego plan: {problem.initial_state.velocity:.2f} m/s", zorder=30)
ax.plot(forecast_xy[:, 0], forecast_xy[:, 1], ":", color="tab:orange",
        label="Traffic forecast (unused by dummy)", zorder=30)
ax.set_xlim(dummy_xy[:, 0].min() - 5, dummy_xy[:, 0].max() + 10)
ax.set_ylim(dummy_xy[:, 1].min() - 6, dummy_xy[:, 1].max() + 5)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9).set_zorder(100)
show_plot(fig)

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.plot(dummy_time, [state.velocity for state in dummy_states], "o-", markersize=3)
ax.set(xlabel="Time ahead [s]", ylabel="Speed [m/s]", title="The dummy keeps its speed")
ax.set_ylim(0, problem.initial_state.velocity * 1.3)
ax.grid(alpha=0.25)
show_plot(fig)


The blue samples follow the route, but the dummy does not use the orange traffic
forecast. It neither brakes for traffic nor checks collisions or road boundaries.
Its heading and steering estimates also require dynamics checks to establish whether
the vehicle can execute them, especially through tight turns.

We therefore need a planner that chooses motion while accounting for predicted
traffic and the vehicle's physical limits. The reactive planner provides this next step.


## 5. Observe reactive planner behavior
The reactive planner generates candidate longitudinal and lateral motions in a
curvilinear frame, converts them to Cartesian trajectories, and checks vehicle
dynamics and collisions with predicted traffic. It selects a candidate using its
internal cost, executes the first few steps, then observes traffic and plans again.

Its adapter builds its own smoothed reference path and uses the predictor from
section 1. SUMO independently simulates how the surrounding vehicles actually move.

This example uses a two-second planning horizon and executes three steps (0.3 seconds)
before replanning. It models a BMW 320i with the kinematic single-track (KS) model.
We reuse `result` for the following plots, animation and export.


In [ ]:
from planner.algorithms.reactive import ReactiveConfig
from planner.runtime.evaluation import describe_result
from planner.runtime.execution import run_spec
from planner.spec import reactive_spec

config = ReactiveConfig(horizon_steps=20, replanning_interval=3)
result = run_spec(starter, reactive_spec(config=config))
describe_result(result)


The summary reports whether the **complete drive** reaches the goal within its time
window and passes dynamics, collision and road-boundary checks. SM1 scores valid
drives, with lower values indicating better performance. This final score is separate
from the planner's internal cost for selecting candidates.

The next plot compares the ego vehicle's executed speed with the dummy's constant-speed
assumption. The dashed line shows that assumption; the reactive curve shows speed
recorded during the SUMO drive.


In [ ]:
driven = result.solution.planning_problem_solutions[0].trajectory.state_list
driven_time = np.array([(state.time_step - current) * scenario.dt for state in driven])
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(driven_time, [state.velocity for state in driven], label="Reactive executed speed")
ax.axhline(problem.initial_state.velocity, linestyle="--", color="gray",
           label="Dummy's constant-speed assumption")
ax.set(xlabel="Simulation time [s]", ylabel="Speed [m/s]", title="Reactive speed over the drive")
ax.grid(alpha=0.25)
ax.legend()
show_plot(fig)

print(f"Replanning calls: {len(result.diagnostics)}")
result.diagnostics[:5]


The animation shows the same completed drive, with the ego vehicle in green and the
goal region in yellow. Use the slider to inspect individual steps. The surrounding
vehicles show their simulated motion, rather than the predictions plotted earlier.


In [ ]:
from planner.common.visualization import animate

animate(result)


### Save the example and continue
Export the ego trajectory as `solution.xml`, the simulated traffic as `scenario.xml`,
and the run information as `metadata.json`. The metadata marks this example as
`in_memory` because we created the planner in the notebook.

[Tutorial 03](03_solution_and_cost_validation.ipynb) shows how to reload and validate
the exported pair. [Tutorial 04](04_reuse_and_evaluate_planners.ipynb) explains how to
run and package your saved submission source.


In [ ]:
from pathlib import Path

from planner.runtime.artifacts import export_result

root = Path.cwd()
if root.name == "tutorials":
    root = root.parent
export_result(result, root / "outputs" / "example-planners")


To use either planner in your submission, activate its factory in
[my_submission/student_planner.py](../my_submission/student_planner.py).
For the custom option, extend the starter with prediction, motion selection and
validation. Your planner only needs to implement `plan(scenario, problem) -> Trajectory`;
the helper methods are optional.

<details>
<summary>Reactive implementation and source</summary>

The [adapter](../planner/algorithms/reactive.py) restores executed steering and
acceleration and refreshes predictions and collision checks. It runs the core planner
in the current process. The supplied `commonroad_rp/` comes from revision `6001c2ca`;
its [manifest](../vendor/reactive_planner.json) records the local correction, and
its [license](../commonroad_rp/LICENSE) is included.

</details>
